# KG1 v157 TODAY - Plano DEFINITIVO HOJE com ZERO regressao

## REGRA ABSOLUTA: NAO REGREDIR ABAIXO DE 0.86

## 4 FASES SEQUENCIAIS (5h total, 2/5 slots Kaggle):

### FASE 0: submit_baseline_v120 (10 min, 1 slot)
Submit V120 adapter direto. Confirma 0.85-0.86 funcional.
**Gate**: score > 0.84 -> CONTINUE.

### FASE 1: diagnostic_local (30 min, 0 slots)
Eval 100 samples local. Classifica erros Tipo A/B/C.

### FASE 2: train_ultraconservative (3-4h, 0 slots)
Warmstart V120 + 11 fixes ULTRA-CONSERVATIVE.

### FASE 3: paired_validate_and_submit (30 min, 1 slot SE PASSAR)
Validacao local pareada vs V120. Submit so se passar GATE.

## CONFIG FINAL:

```
Adapter: /content/v120_adapter (3.55 GB, r=32 alpha=32, score 0.85-0.86)
LR: 1e-5 cosine
max_steps: 80 (NAO epoch unit)
warmup: 10%
batch_effective: 16
lora_dropout: 0.0 (preserva V120)
label_smoothing: 0.0 (math)
weight_decay: 0.0
max_grad_norm: 0.5
max_seq: 6144 (truncation preserve \boxed{})
attn: flash_attention_2
neftune: alpha=5
seed: 42
```

## Probabilidades:

- FASE 0+1+2+3 completo: 30-40% prob 0.87+
- Risco regressao: <5% (gates absolutos)
- Manter 0.86: >95%

## Setup:

1. Runtime A100 ALTA RAM 80GB
2. Colab Secrets: HF_KEY, KAGGLE_USERNAME, KAGGLE_KEY
3. **PRE-REQUISITO**: /content/v120_adapter ja baixado
4. Form param FASE escolher e Run all


In [ ]:
# CELL UNICA v157 TODAY - 4 fases com gates anti-regressao
import os, sys, time, math, gc, glob, csv, json, urllib.request, statistics, subprocess, shutil, re

# === FORM PARAMS ===
FASE = "0_submit_baseline"  #@param ["0_submit_baseline", "1_diagnostic", "2_train_ultraconservative", "3_paired_validate_submit"]
ADAPTER_PATH = "/content/v120_adapter"  #@param {type:"string"}
DRY_RUN = False  #@param {type:"boolean"}

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print('=' * 70)
print(f'KG1 v157 TODAY - FASE {FASE}')
print('=' * 70)

# ============================================================
# COMMON SETUP (todas as fases)
# ============================================================
print('\n[SETUP] Secrets + GPU + verify adapter')
try:
    from google.colab import userdata, drive
    hf_token = None
    for k in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(k)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                break
        except: pass
    if not hf_token:
        raise RuntimeError('HF_KEY missing')
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except: pass
except ImportError:
    raise RuntimeError('Run in Colab')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')

# Verify adapter
adapter_dir = None
if os.path.exists(ADAPTER_PATH):
    for root, dirs, files in os.walk(ADAPTER_PATH):
        if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
            sz = os.path.getsize(os.path.join(root, 'adapter_model.safetensors'))
            if sz > 1_000_000_000:
                adapter_dir = root
                break
if not adapter_dir:
    raise RuntimeError(f'Adapter not found in {ADAPTER_PATH}. Download first!')

with open(os.path.join(adapter_dir, 'adapter_config.json')) as f:
    cfg = json.load(f)
print(f'  ADAPTER: {adapter_dir}')
print(f'  r={cfg.get("r")} alpha={cfg.get("lora_alpha")} dropout={cfg.get("lora_dropout")}')

# ============================================================
# FASE 0: submit_baseline_v120
# ============================================================
if FASE == '0_submit_baseline':
    print('\n[FASE 0] Submit V120 baseline (NO TRAIN, ~10 min)')

    # Install minimal kaggle
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle'], capture_output=True)

    SUBMIT_DIR = '/content/submit_v157_baseline'
    os.makedirs(SUBMIT_DIR, exist_ok=True)
    for fname in ['adapter_config.json', 'adapter_model.safetensors']:
        src = os.path.join(adapter_dir, fname)
        if os.path.exists(src):
            shutil.copy(src, SUBMIT_DIR)
            print(f'  Copied {fname}: {os.path.getsize(src)/1e6:.1f} MB')

    zip_path = '/content/v157_baseline.zip'
    subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)
    sz_mb = os.path.getsize(zip_path) / 1e6
    print(f'  ZIP: {sz_mb:.1f} MB')

    if not DRY_RUN:
        msg = 'v157 FASE 0 - V120 Modal Surfer baseline (expected 0.85-0.86 confirmed)'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=600)
        print(f'  Submit: {r.stdout[:300]}{r.stderr[:300]}')
        print('\n  >>> SLOTS USED: 1/5')
        print('  >>> Aguarde ~15-30 min para Kaggle reportar score')
        print('  >>> Se score > 0.84: CONTINUE para FASE 1')
        print('  >>> Se score < 0.84: investigar adapter antes de prosseguir')

    print('\n' + '=' * 70)
    print('FASE 0 DONE. Aguarde Kaggle score, depois rode FASE 1')
    print('=' * 70)
    raise SystemExit(0)

# ============================================================
# Setup pesado para FASE 1, 2, 3
# ============================================================
print('\n[DEPS] Install full deps (FASE 1+)')
deps = [
    'transformers>=5.3.0', 'peft>=0.14.0', 'trl>=0.14.0', 'datasets>=3.0.0',
    'accelerate>=1.3.0', 'bitsandbytes>=0.45.0', 'huggingface_hub>=0.27.0',
    'torchao>=0.16.0', 'kaggle', 'einops', 'packaging', 'ninja', 'triton',
    'flash-attn>=2.7.0',
]
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=400)
    print(f'  {"[OK]" if r.returncode == 0 else "[FAIL]"} {pkg}')
for m in ['transformers', 'peft', 'trl', 'bitsandbytes', 'torchao']:
    if m in sys.modules:
        del sys.modules[m]

print('\n[MAMBA] Install mamba-ssm 2.3.1')
import torch
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'

attempts = [('cu12', torch_short, abi_str)]
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        if (('cu12', t, abi)) not in attempts:
            attempts.append(('cu12', t, abi))

for cu, t, abi in attempts:
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    ok_cc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', cc_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    ok_mb = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', mb_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    if ok_cc and ok_mb:
        print(f'  [OK] {cu} torch{t} abi{abi}')
        break

for m in ['mamba_ssm', 'causal_conv1d']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# Load Model + V120 Adapter (comum a 1, 2, 3)
# ============================================================
print(f'\n[LOAD] Nemotron-30B + V120 adapter (FlashAttn2)')
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

t0 = time.time()
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=True, token=hf_token,
        attn_implementation='flash_attention_2',
    )
except Exception as e:
    print(f'  FlashAttn2 fail: {e}, fallback eager')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=True, token=hf_token, attn_implementation='eager',
    )
model.config.use_cache = (FASE in ('1_diagnostic', '3_paired_validate_submit'))
print(f'  [OK] Model {time.time()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

is_trainable = (FASE == '2_train_ultraconservative')
model = PeftModel.from_pretrained(model, adapter_dir, adapter_name='default',
                                   token=hf_token, is_trainable=is_trainable)

# ============================================================
# FASE 1: diagnostic_local
# ============================================================
if FASE == '1_diagnostic':
    print('\n[FASE 1] Diagnostic local 100 samples (~30 min)')
    model.eval()

    KAGGLE_REGEX = re.compile(r'\\boxed\{([^}]*)(?:\}|$)')
    def kaggle_verify(stored, predicted):
        stored = stored.strip(); predicted = predicted.strip()
        if re.fullmatch(r'[01]+', stored):
            return predicted.lower() == stored.lower()
        try:
            return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
        except:
            return predicted.lower() == stored.lower()

    DATA_PATH = '/content/cryptarithm_813.jsonl'
    if not os.path.exists(DATA_PATH):
        url = 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl'
        urllib.request.urlretrieve(url, DATA_PATH)
    samples = [json.loads(l) for l in open(DATA_PATH, encoding='utf-8') if json.loads(l).get('is_valid')]

    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    results = {'total': 0, 'correct': 0, 'tipo_a': 0, 'tipo_b': 0, 'tipo_c': 0, 'examples_c': []}

    N = 100
    t0 = time.time()
    for i, row in enumerate(samples[:N]):
        if i % 10 == 0 and i > 0:
            elapsed = time.time() - t0
            print(f'  [{i}/{N}] {elapsed:.0f}s')
        prompt = (row.get('prompt') or '').strip()
        gold = (row.get('answer') or '').strip()
        if not prompt or not gold:
            continue
        msgs = [{'role': 'user', 'content': prompt + PROMPT_SUFFIX}]
        inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt',
                                                add_generation_prompt=True, enable_thinking=True).to('cuda')
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=2048, temperature=0.0, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

        results['total'] += 1
        ms = KAGGLE_REGEX.findall(text)
        if not ms:
            results['tipo_a'] += 1
            continue
        extracted = next((m.strip() for m in reversed(ms) if m.strip()), None)
        if not extracted:
            results['tipo_a'] += 1
            continue
        if kaggle_verify(gold, extracted):
            results['correct'] += 1
            continue
        # Tipo B vs C
        cleanups = [
            extracted.replace('=', ' ').strip().split()[-1] if extracted.split() else '',
            re.sub(r'\s+', '', extracted),
            re.sub(r'[a-zA-Z/\s]', '', extracted),
            extracted.rstrip('.,;'),
        ]
        is_c = False
        for a in cleanups:
            if a and a != extracted and kaggle_verify(gold, a):
                is_c = True
                if len(results['examples_c']) < 5:
                    results['examples_c'].append({'gold': gold[:40], 'ext': extracted[:60], 'fix': a[:40]})
                break
        if is_c:
            results['tipo_c'] += 1
        else:
            results['tipo_b'] += 1

    print(f'\n=== FASE 1 RESULTS ===')
    tot = max(1, results['total'])
    print(f'Adapter: V120 (size {os.path.getsize(adapter_dir+"/adapter_model.safetensors")/1e9:.1f} GB)')
    print(f'Total: {results["total"]} | Time: {time.time()-t0:.0f}s')
    print(f'Kaggle parser score: {results["correct"]}/{results["total"]} = {100*results["correct"]/tot:.1f}%')
    print(f'Tipo A (no boxed):    {results["tipo_a"]:3} ({100*results["tipo_a"]/tot:.1f}%)')
    print(f'Tipo B (math wrong):  {results["tipo_b"]:3} ({100*results["tipo_b"]/tot:.1f}%)')
    print(f'Tipo C (format off):  {results["tipo_c"]:3} ({100*results["tipo_c"]/tot:.1f}%)')
    if results['examples_c']:
        print(f'\nExamples Tipo C:')
        for ex in results['examples_c']:
            print(f'  gold="{ex["gold"]}" ext="{ex["ext"]}" fix="{ex["fix"]}"')

    print(f'\n=== DECISAO ===')
    pct_c = 100 * results['tipo_c'] / tot
    pct_b = 100 * results['tipo_b'] / tot
    pct_a = 100 * results['tipo_a'] / tot
    if pct_c >= 5:
        print(f'>>> HIPOTESE PARSER MISS CONFIRMADA ({pct_c:.1f}% Tipo C)')
        print(f'>>> NEXT: FASE 2 com FORMAT-strict focus (esperado +0.005-0.015 LB)')
    elif pct_b > pct_c * 2:
        print(f'>>> Erros majoritariamente MATH ({pct_b:.1f}% Tipo B)')
        print(f'>>> NEXT: FASE 2 com HEM math focus')
    elif pct_a >= 20:
        print(f'>>> Truncation issue ({pct_a:.1f}% Tipo A)')
        print(f'>>> Investigar inference settings antes de FASE 2')
    else:
        print(f'>>> Resultado balanceado - FASE 2 com HEM mixed')

    with open('/content/v157_diagnostic.json', 'w') as f:
        json.dump(results, f, indent=2)

    print('\n=== FASE 1 DONE - Run FASE 2 next ===')
    raise SystemExit(0)

# ============================================================
# FASE 2: train_ultraconservative
# ============================================================
if FASE == '2_train_ultraconservative':
    print('\n[FASE 2] Train ULTRA-CONSERVATIVE (lr=1e-5, 80 steps, batch=16, ls=0.0)')
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': True})

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B')

    # Datasets
    DATA_DIR = '/content/data'
    os.makedirs(DATA_DIR, exist_ok=True)
    URLS = {
        'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
        'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
        'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
    }
    for name, url in URLS.items():
        out = os.path.join(DATA_DIR, name)
        if not os.path.exists(out):
            urllib.request.urlretrieve(url, out)

    from datasets import Dataset
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    all_data = []
    for fname, src in [('cryptarithm_813.jsonl', 'crypt'), ('eq_guess_150.jsonl', 'eq'), ('synth_4425.jsonl', 'synth')]:
        with open(os.path.join(DATA_DIR, fname), encoding='utf-8') as f:
            for line in f:
                row = json.loads(line)
                p = (row.get('prompt') or '').strip()
                c = (row.get('cot') or row.get('generated_cot') or '').strip()
                valid = row.get('is_valid', True)
                if p and c and valid:
                    all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c, 'src': src})
    ORDER = {'eq': 0, 'crypt': 1, 'synth': 2}
    all_data.sort(key=lambda x: ORDER.get(x['src'], 99))
    ds = Dataset.from_list(all_data)
    print(f'  Total: {len(ds)} samples')

    # Tokenize com truncation preserve final
    MAX_LENGTH = 6144
    def fmt(ex):
        msgs = [{'role': 'user', 'content': ex['prompt']},
                {'role': 'assistant', 'content': ex['completion']}]
        full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        prm = tokenizer.apply_chat_template([msgs[0]], tokenize=False, add_generation_prompt=True, enable_thinking=True)
        full_ids = tokenizer(full, add_special_tokens=False)['input_ids']
        prm_ids = tokenizer(prm, add_special_tokens=False)['input_ids']
        if len(full_ids) > MAX_LENGTH:
            prefix = min(len(prm_ids), MAX_LENGTH // 3)
            suffix = MAX_LENGTH - prefix
            full_ids = full_ids[:prefix] + full_ids[-suffix:]
        labels = list(full_ids)
        n_prompt = min(len(prm_ids), len(labels))
        for i in range(n_prompt):
            labels[i] = -100
        return {'input_ids': full_ids, 'attention_mask': [1]*len(full_ids), 'labels': labels}

    tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
    seq_lens = [len(tokenized[i]['input_ids']) for i in range(min(500, len(tokenized)))]
    print(f'  Lens: min={min(seq_lens)} median={statistics.median(seq_lens):.0f} max={max(seq_lens)}')

    PAD_ID = tokenizer.pad_token_id or tokenizer.eos_token_id
    def collator(features):
        max_len = max(len(f['input_ids']) for f in features)
        max_len = ((max_len + 7) // 8) * 8
        ids, masks, lbls = [], [], []
        for f in features:
            pad = max_len - len(f['input_ids'])
            ids.append(f['input_ids'] + [PAD_ID]*pad)
            masks.append(f['attention_mask'] + [0]*pad)
            lbls.append(f['labels'] + [-100]*pad)
        return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(masks), 'labels': torch.tensor(lbls)}

    from transformers import Trainer, TrainingArguments
    OUT = '/content/output_v157'
    os.makedirs(OUT, exist_ok=True)

    train_args = TrainingArguments(
        output_dir=OUT,
        max_steps=80,                          # GPT-5.5: 80-120 fixed
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,        # batch effective 16
        learning_rate=1e-5,                    # GPT-5.5 ultraconservador
        lr_scheduler_type='cosine',
        warmup_steps=8,                        # 10%
        adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
        weight_decay=0.0,                      # GPT-5.5
        max_grad_norm=0.5,                     # GPT-5.5
        label_smoothing_factor=0.0,            # GPT-5.5+Gemini Pro
        logging_steps=5,
        save_steps=20,
        save_total_limit=4,
        bf16=True,
        optim='adamw_torch_fused',
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': True},
        remove_unused_columns=False, report_to='none',
        dataloader_num_workers=0, seed=42,
        neftune_noise_alpha=5,                 # Gemini Pro
    )

    trainer = Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=collator)
    print(f'  Steps: 80 | Warmup: 8 | Batch eff: 16 | LR: 1e-5 | LS: 0.0 | Dropout: preserved')

    t0 = time.time()
    trainer.train()
    train_min = (time.time() - t0) / 60
    print(f'\n[OK] Training: {train_min:.1f} min')

    ADAPTER_OUT = f'{OUT}/final_adapter'
    trainer.save_model(ADAPTER_OUT)
    tokenizer.save_pretrained(ADAPTER_OUT)
    print(f'  Saved: {ADAPTER_OUT}')

    # Backup GDrive
    try:
        GD = '/content/drive/My Drive/KG1_v157_trained_adapter'
        if os.path.exists(GD): shutil.rmtree(GD)
        shutil.copytree(ADAPTER_OUT, GD)
        print(f'  GDrive backup: {GD}')
    except Exception as e:
        print(f'  GDrive: {e}')

    print('\n=== FASE 2 DONE - Run FASE 3 (paired_validate_submit) ===')
    raise SystemExit(0)

# ============================================================
# FASE 3: paired_validate_and_submit
# ============================================================
if FASE == '3_paired_validate_submit':
    print('\n[FASE 3] Paired validation + Submit (NO-GO gate)')

    TRAINED_PATH = '/content/output_v157/final_adapter'
    if not os.path.exists(TRAINED_PATH):
        # Try GDrive
        gd = '/content/drive/My Drive/KG1_v157_trained_adapter'
        if os.path.exists(gd):
            shutil.copytree(gd, TRAINED_PATH)
        else:
            raise RuntimeError('Trained adapter not found. Run FASE 2 first.')

    print(f'  Loading TRAINED adapter from {TRAINED_PATH}...')

    # Load trained adapter on top
    model.load_adapter(TRAINED_PATH, adapter_name='trained')

    # Eval helper
    KAGGLE_REGEX = re.compile(r'\\boxed\{([^}]*)(?:\}|$)')
    def kaggle_verify(stored, predicted):
        stored = stored.strip(); predicted = predicted.strip()
        if re.fullmatch(r'[01]+', stored):
            return predicted.lower() == stored.lower()
        try:
            return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
        except:
            return predicted.lower() == stored.lower()

    def eval_adapter(samples, adapter_name, max_n=100):
        model.set_adapter(adapter_name)
        model.eval()
        correct = 0
        total = 0
        PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
        for row in samples[:max_n]:
            prompt = (row.get('prompt') or '').strip()
            gold = (row.get('answer') or '').strip()
            if not prompt or not gold: continue
            msgs = [{'role': 'user', 'content': prompt + PROMPT_SUFFIX}]
            inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt',
                                                    add_generation_prompt=True, enable_thinking=True).to('cuda')
            with torch.no_grad():
                out = model.generate(inputs, max_new_tokens=2048, temperature=0.0, do_sample=False,
                                      pad_token_id=tokenizer.eos_token_id)
            text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
            ms = KAGGLE_REGEX.findall(text)
            if not ms:
                total += 1; continue
            extracted = next((m.strip() for m in reversed(ms) if m.strip()), None)
            total += 1
            if extracted and kaggle_verify(gold, extracted):
                correct += 1
        return correct, total

    # Load 100 samples
    DATA_PATH = '/content/cryptarithm_813.jsonl'
    if not os.path.exists(DATA_PATH):
        url = 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl'
        urllib.request.urlretrieve(url, DATA_PATH)
    samples = [json.loads(l) for l in open(DATA_PATH, encoding='utf-8') if json.loads(l).get('is_valid')]

    print('  Eval V120 baseline (100 samples)...')
    t0 = time.time()
    base_correct, base_total = eval_adapter(samples, 'default', max_n=100)
    print(f'  V120 baseline: {base_correct}/{base_total} = {100*base_correct/max(1,base_total):.1f}% [{time.time()-t0:.0f}s]')

    print('  Eval TRAINED (100 samples)...')
    t0 = time.time()
    trained_correct, trained_total = eval_adapter(samples, 'trained', max_n=100)
    print(f'  TRAINED:       {trained_correct}/{trained_total} = {100*trained_correct/max(1,trained_total):.1f}% [{time.time()-t0:.0f}s]')

    delta = trained_correct - base_correct
    print(f'\n  DELTA: {delta:+d} ({100*delta/max(1,base_total):+.1f}%)')

    # === NO-GO GATE ===
    print(f'\n  === NO-GO GATE ===')
    if delta < 0:
        print(f'  >>> ❌ REGRESSION DETECTED ({delta} fewer correct)')
        print(f'  >>> ABORT submit. KEEP V120 baseline.')
        print(f'  >>> Probabilidade Kaggle regredir: HIGH')
        print(f'  >>> RECOMENDACAO: descartar candidate, investigar')
    elif delta == 0:
        print(f'  >>> ⚠️ NO IMPROVEMENT (tie)')
        print(f'  >>> RISCO Kaggle: pode subir/cair por variance ±1 acerto')
        print(f'  >>> DECISAO: voce escolhe submit ou nao')
    else:
        print(f'  >>> ✅ IMPROVEMENT ({delta} more correct = +{100*delta/100:.1f}%)')
        print(f'  >>> SUBMIT recommended')

    if delta >= 0 and not DRY_RUN:
        # Submit candidate
        SUBMIT_DIR = '/content/submit_v157_trained'
        os.makedirs(SUBMIT_DIR, exist_ok=True)
        for fname in ['adapter_config.json', 'adapter_model.safetensors']:
            src = os.path.join(TRAINED_PATH, fname)
            if os.path.exists(src):
                shutil.copy(src, SUBMIT_DIR)

        zip_path = '/content/v157_trained.zip'
        subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)

        msg = f'v157 train_ultraconservative warmstart V120 - lr=1e-5 80steps batch=16 - paired delta={delta:+d}'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=600)
        print(f'\n  Submit: {r.stdout[:200]}{r.stderr[:200]}')
        print(f'  >>> SLOTS USED: 2/5 (FASE 0 + FASE 3)')
    else:
        if DRY_RUN:
            print(f'  >>> DRY_RUN mode - would submit if delta >= 0')
        else:
            print(f'  >>> NOT submitting (regression)')

    print('\n' + '=' * 70)
    print('FASE 3 DONE')
    print('=' * 70)
    raise SystemExit(0)

print('\n[ERROR] Unknown FASE: ' + str(FASE))
